## 15 - Gen AI Report Generator

This notebook uses Groq's Llama 3 model to automatically generate
professional analyst-style reports for each earnings call.

### What it does?
1. Pulls sentiment scores, hedge ratios and stock returns from database
2. Builds a structured prompt with all the data
3. Sends to Groq Llama 3 via API
4. Generates a 4-paragraph professional analyst report
5. Saves each report as a markdown file

### Output
74 professional analyst reports saved to reports/company_reports/
One report per company per quarter — ready to share on GitHub.

### Technology
- Groq API — free, fast, no quota issues
- Llama 3.1 8B — powerful open source language model
- Markdown output — renders beautifully on GitHub

In [11]:
# Import libraries
from groq import Groq
from dotenv import load_dotenv
import sqlite3
import pandas as pd
import os

# Load API key from .env file
load_dotenv()
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

# Configure Groq
client = Groq(api_key=GROQ_API_KEY)

# Test connection
response = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[{"role": "user", "content": "Say 'Groq connected successfully!' and nothing else."}]
)
print(response.choices[0].message.content)

Groq connected successfully!


In [8]:
# Connect to database
conn = sqlite3.connect("../data/earnings_research.db")
df = pd.read_sql("SELECT * FROM master_analysis", conn)
df = df[df["avg_compound"] > 0].copy()

# Create reports folder
os.makedirs("../reports/company_reports", exist_ok=True)

print(f"Loaded {len(df)} records")

# Function to generate analyst report
def generate_report(ticker, quarter, df, client):
    # Get data for this company
    row = df[(df["ticker"] == ticker) & (df["quarter"] == quarter)]
    
    if len(row) == 0:
        return f"No data found for {ticker} {quarter}"
    
    row = row.iloc[0]
    
    # Determine signals
    sentiment_signal = "HIGH" if row["avg_compound"] > 0.28 else "MODERATE" if row["avg_compound"] > 0.20 else "LOW"
    hedge_signal = "HIGH" if row["hedge_ratio"] > 0.014 else "MODERATE" if row["hedge_ratio"] > 0.010 else "LOW"
    performance = "OUTPERFORMED" if row["return_90d"] > 0 else "UNDERPERFORMED"
    verdict = "OVERCONFIDENCE TRAP" if row["avg_compound"] > 0.28 and row["return_90d"] < 0 else \
              "HONEST & ACCURATE" if row["avg_compound"] < 0.20 and row["return_90d"] > 0 else \
              "CONFIDENT & JUSTIFIED" if row["avg_compound"] > 0.28 and row["return_90d"] > 0 else "MIXED SIGNAL"
    
    # Build prompt
    prompt = f"""You are a senior financial analyst. Write a professional earnings call analysis report for {ticker} {quarter}.

Here is the data:
- Company: {ticker}
- Quarter: {quarter}
- Market Period: {row["period"]}
- VADER Sentiment Score: {row["avg_compound"]} (scale -1 to +1)
- FinBERT Financial Sentiment: {row["finbert_score"]} (scale -1 to +1)
- Hedge Ratio: {row["hedge_ratio"]} (proportion of cautious words)
- Sentiment Signal: {sentiment_signal}
- Hedging Signal: {hedge_signal}
- 30-day Stock Return: {row["return_30d"]}%
- 60-day Stock Return: {row["return_60d"]}%
- 90-day Stock Return: {row["return_90d"]}%
- Performance: {performance}
- Verdict: {verdict}

Write a 4-paragraph professional analyst report covering:
1. Executive Summary - what the CEO's language revealed
2. Sentiment Analysis - what the scores mean in context
3. Stock Performance - what actually happened
4. Conclusion - was the CEO's confidence justified?

Use professional financial language. Be specific about the numbers. Keep it factual and insightful."""

    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=600
    )
    
    return response.choices[0].message.content

# Test on Apple Q1 2020
print("Generating report for AAPL Q1 2020...")
report = generate_report("AAPL", "Q1_2020", df, client)
print(report)

Loaded 74 records
Generating report for AAPL Q1 2020...
**Earnings Call Analysis Report: AAPL Q1 2020**

**Executive Summary**

During Apple's Q1 2020 earnings call, CEO Tim Cook's language suggested optimistic confidence in the company's performance. Although the company reported a decline in iPhone sales, Cook emphasized the growth prospects of services, Wearables, and iPad sales. Notably, he stated, "We're proud of the fact that our Services business exceeded the quarter's revenue guidance by $1.6 billion." This statement reveals a subtle yet significant emphasis on the positive aspects of the company's performance, while downplaying the iPhone sales decline. Such selective emphasis may indicate a degree of overconfidence, particularly given the challenges posed by the COVID-19 pandemic at the time.

**Sentiment Analysis**

Our quantitative analysis confirms the CEO's apparent optimism. The VADER Sentiment Score of 0.2968 indicates a net positive sentiment, while the FinBERT Financi

In [9]:
# Generate reports for all 74 companies
success = []
failed = []

for _, row in df.iterrows():
    ticker = row["ticker"]
    quarter = row["quarter"]
    
    try:
        report = generate_report(ticker, quarter, df, client)
        
        # Create markdown file
        filename = f"../reports/company_reports/{ticker}_{quarter}_report.md"
        
        with open(filename, "w", encoding="utf-8") as f:
            f.write(f"# {ticker} {quarter} Earnings Call Analysis\n\n")
            f.write(f"**Market Period:** {row['period']}\n\n")
            f.write(f"**Date:** {row['date']}\n\n")
            f.write("---\n\n")
            f.write(report)
            f.write("\n\n---\n\n")
            f.write("*Generated by CEO Language Analyzer — earnings-sentiment-analysis project*\n")
        
        success.append(f"{ticker} {quarter}")
        print(f"✓ {ticker} {quarter}")
    
    except Exception as e:
        failed.append(f"{ticker} {quarter}")
        print(f"✗ {ticker} {quarter}: {e}")

print(f"\nSuccessfully generated: {len(success)}")
print(f"Failed: {len(failed)}")
if failed:
    for f in failed:
        print(f"  - {f}")

✓ AAPL Q1_2020
✓ ABT Q1_2020
✓ AMD Q1_2020
✓ BAC Q1_2020
✓ BLK Q1_2020
✓ COST Q1_2020
✓ CRM Q1_2020
✓ CVX Q1_2020
✓ DHR Q1_2020
✓ GOOGL Q1_2020
✓ GS Q1_2020
✓ HAL Q1_2020
✓ HD Q1_2020
✓ INTC Q1_2020
✓ KO Q1_2020
✓ MCD Q1_2020
✓ META Q1_2020
✓ MRK Q1_2020
✓ MS Q1_2020
✓ NKE Q1_2020
✓ OXY Q1_2020
✓ PFE Q1_2020
✓ PG Q1_2020
✓ SBUX Q1_2020
✓ SLB Q1_2020
✓ TGT Q1_2020
✓ TMO Q1_2020
✓ UNH Q1_2020
✓ VLO Q1_2020
✓ WFC Q1_2020
✓ WMT Q1_2020
✓ JNJ Q1_2020
✓ JPM Q1_2020
✓ MSFT Q1_2020
✓ AAPL Q2_2020
✓ AMZN Q1_2020
✓ XOM Q1_2020
✓ JPM Q2_2020
✓ SLB Q2_2021
✓ MSFT Q4_2022
✓ AAPL Q4_2022
✓ MSFT Q2_2023
✓ ABT Q2_2023
✓ AMD Q2_2023
✓ BAC Q2_2023
✓ BLK Q2_2023
✓ COST Q2_2023
✓ CRM Q2_2023
✓ CVX Q2_2023
✓ DHR Q2_2023
✓ GOOGL Q2_2023
✓ GS Q2_2023
✓ HAL Q2_2023
✓ HD Q2_2023
✓ INTC Q2_2023
✓ KO Q2_2023
✓ MCD Q2_2023
✓ META Q2_2023
✓ MRK Q2_2023
✓ MS Q2_2023
✓ NKE Q2_2023
✓ OXY Q2_2023
✓ PFE Q2_2023
✓ PG Q2_2023
✓ SBUX Q2_2023
✓ TGT Q2_2023
✓ TMO Q2_2023
✓ UNH Q2_2023
✓ VLO Q2_2023
✓ WFC Q2_2023
✓ WMT Q2_20

In [10]:
conn.close()
print("All 74 reports saved to reports/company_reports/")
print("Database connection closed.")

All 74 reports saved to reports/company_reports/
Database connection closed.
